# Figures 1 and 2 — matched connectivity, and scaling the dense network

**Figure 1.** Block-sparse vs dense at a matched budget of ~19,000 weights on
4-task multi-MNIST, stationary (left) and non-stationary (right), 30 seeds.

**Figure 2.** The dense network at seven budgets, 1x to 64x of that base, with
the 19,000-weight block-sparse network as a fixed reference line.

Figure 2 is where the two regimes come apart. In the stationary problem the
dense network catches up once it has ~4x the parameters, because with enough
updates the average over the final 10% of steps is dominated by converged
performance, which a large enough network reaches regardless of how slowly it
gets there. In the non-stationary problem nothing converges, so the same
average measures the *speed* of adaptation — and there the dense network never
catches up, and past 152,000 parameters it gets worse.

Produced by `experiments/sweeps/01_matched_and_scaling/best/`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from multi_mnist.analysis import (
    asymptotic_metric, fetch_per_seed_arrays, filter_sweep, filter_to_best,
    get_color_palette, load_export, normalize_columns, plot_ci_curve,
    plot_sensitivity, save_fig, set_style, stack_per_seed,
)

%matplotlib inline


In [ ]:
# The Comet project the published runs live in. Change it if you
# re-run the sweeps into your own project.
PROJECT = 'paper-weight-pruning-scaling-best'

# The sweeps this notebook reads. Each entry lists every Comet sweep name that
# belongs to the same logical experiment, so re-runs and addenda merge.
SWEEPS = {
    ('stationary', 'Block-Sparse'):    ['01_block_sparse_stationary_best'],
    ('stationary', 'Dense'):           ['01_dense_stationary_best'],
    ('nonstationary', 'Block-Sparse'): ['02_block_sparse_nonstationary_best'],
    ('nonstationary', 'Dense'):        ['02_dense_nonstationary_best'],
}

# Parameters that identify one experimental cell. Comet retries land on the
# same cell and are deduped against these.
CELL_COLS = ['model.init_strategy', 'model.initial_hidden_units',
             'seed_offset', 'optimizer.learning_rate']

# Stationary runs are scored on held-out test accuracy; non-stationary runs
# have no test split, so they are scored on online accuracy.
METRIC = {'stationary': 'test_accuracy', 'nonstationary': 'accuracy'}

PALETTE = get_color_palette(classes=['Block-Sparse', 'Dense'])
BUDGETS = [6, 12, 24, 48, 96, 192, 384]   # dense hidden units, 1x .. 64x


In [ ]:
# Run once to fetch the sweep results from Comet, then leave it commented out.
# from multi_mnist.analysis import download_project
# download_project(PROJECT, output_dir='data')


In [ ]:
cfg_all, run_all = load_export(PROJECT, data_dir='data')
cfg_all = normalize_columns(cfg_all)
print(f'{len(cfg_all)} trials, {len(run_all)} metric rows')
print(sorted(cfg_all['sweep_name'].dropna().unique()))


## Figure 1 — matched budget, 30 seeds

Both networks hold ~19,056 weights: block-sparse as 24 units split into 4
independent sub-networks of 6, dense as 6 fully-connected units. The band is a
95% confidence interval over 30 seeds, computed from the per-seed arrays; at
this seed count it is too narrow to see, which is itself worth reporting.


In [ ]:
def matched_curves(regime, budget=None):
    """Per-seed curves for each method at the matched budget, best step-size."""
    out = {}
    for (reg, method), names in SWEEPS.items():
        if reg != regime:
            continue
        cfg, run = filter_sweep(cfg_all, run_all, names, cell_cols=CELL_COLS)
        if method == 'Dense' and budget is not None:
            cfg = cfg[cfg['model.initial_hidden_units'] == budget]
            run = run[run['run_id'].isin(cfg['run_id'])]
        # Both 15-seed halves of the same cell must come along, so group by
        # everything except seed_offset.
        run_ids = cfg['run_id'].tolist()
        assets = fetch_per_seed_arrays(run_ids, cache_dir='data/per_seed_cache')
        losses, accs = stack_per_seed(assets, run_ids)
        if accs is not None:
            out[method] = accs
    return out


def plot_figure_1(regime, title, savename):
    set_style('4-col')
    fig, ax = plt.subplots()
    for method, accs in matched_curves(regime, budget=6).items():
        steps = np.arange(accs.shape[0]) * 1000
        plot_ci_curve(steps, accs, label=method, color=PALETTE[method], ax=ax)
    ax.set_xlabel('Step'); ax.set_ylabel('Accuracy')
    ax.set_title(title); ax.set_ylim(0, 1.0)
    ax.ticklabel_format(axis='x', style='sci', scilimits=(0, 0), useMathText=True)
    fig.legend(loc='upper right')
    save_fig(savename, fig_dir='../figures/generated')
    plt.show()


plot_figure_1('stationary', 'Stationary', 'fig1_stationary')
plot_figure_1('nonstationary', 'Non-Stationary', 'fig1_nonstationary')


## Figure 2 — dense scaling against a fixed block-sparse reference

Each dense point is that budget at its own best step-size. The dotted line is
the 19,000-weight block-sparse network, which does not move.


In [ ]:
def plot_figure_2(regime, title, savename):
    metric = METRIC[regime]
    dense_cfg, dense_run = filter_sweep(
        cfg_all, run_all, SWEEPS[(regime, 'Dense')], cell_cols=CELL_COLS)
    bs_cfg, bs_run = filter_sweep(
        cfg_all, run_all, SWEEPS[(regime, 'Block-Sparse')], cell_cols=CELL_COLS)

    dense = asymptotic_metric(
        dense_cfg, dense_run, metric, ['model.initial_hidden_units'], tail_frac=0.10)
    dense = dense.groupby('model.initial_hidden_units')['_metric'].max()
    bs = asymptotic_metric(
        bs_cfg, bs_run, metric, ['model.init_strategy'], tail_frac=0.10)['_metric'].max()

    set_style('4-col')
    fig, ax = plt.subplots()
    x = np.arange(len(BUDGETS))
    ax.plot(x, [dense.get(b, np.nan) for b in BUDGETS], '-o',
            color=PALETTE['Dense'], label='Dense')
    ax.axhline(bs, linestyle='--', color=PALETTE['Block-Sparse'], label='Block-Sparse')
    ax.set_xticks(x)
    ax.set_xticklabels([f'{2**i}x' for i in range(len(BUDGETS))])
    ax.set_xlabel('Budget (x 19K Weights)'); ax.set_ylabel('Accuracy')
    ax.set_title(title); ax.set_ylim(0, 1.0); ax.grid(True, alpha=0.4)
    fig.legend(loc='lower right')
    save_fig(savename, fig_dir='../figures/generated')
    plt.show()


plot_figure_2('stationary', 'Stationary Scaling', 'fig2_stationary_scaling')
plot_figure_2('nonstationary', 'Non-Stationary Scaling', 'fig2_nonstationary_scaling')


## Step-size sensitivity (not in the paper)

A check that the swept step-size grid brackets the optimum at every budget,
rather than running into the edge of the grid. Reads the `sweep/` configs.


In [ ]:
SWEEP_# The Comet project the published runs live in. Change it if you
# re-run the sweeps into your own project.
PROJECT = 'paper-weight-pruning-scaling-sweep'

try:
    s_cfg, s_run = load_export(SWEEP_PROJECT, data_dir='data')
    s_cfg = normalize_columns(s_cfg)
    for regime, names in [('stationary', '01_dense_stationary_lr'),
                          ('nonstationary', '02_dense_nonstationary_lr')]:
        c, r = filter_sweep(s_cfg, s_run, names, cell_cols=CELL_COLS)
        asym = asymptotic_metric(c, r, METRIC[regime], ['model.initial_hidden_units'])
        set_style('2-col')
        fig, ax = plt.subplots()
        plot_sensitivity(asym, ax=ax, hue_col='model.initial_hidden_units',
                         title=f'Dense, {regime}', ylabel='Accuracy',
                         ylim=(0, 1), legend_title='Hidden units')
        plt.show()
except FileNotFoundError:
    print(f'No export for {SWEEP_PROJECT}; run the sweep/ configs first.')
